# Exercise 6

This exercise is based on Chapter 4 (Spatial Weights) of the Geographic Data Science book.

The material can be found in: `GSP538/gds_book/notebooks/04_spatial_weights.ipynb`

#### Notes on Textbook

- This chapter is all about spatial relationships. You have seen most of the concepts discussed in this chapter in previous GIS courses. The additional value here is translating those general concepts into a formal structure. The chapter and this exercise is asking you to refine your existing knowledge to be able to speak precisely about spatial relationships.
- It is not necessary that you understand the code to create the 3x3 grid, but it does drive home the idea that polygons in a GIS are simply made up of points.

- If some of the cells don't run, be sure you're in the `gsp538` environment
- VS Code allows you to have side-by-side notebooks (e.g., this exercise and the book chapter).
  - Collapse the Explorer by clicking the little icon in the upper-left of VS Code
  - Click and drag the tab of the notebook you want to reposition (e.g. drag it to the right side of the VS Code window)
- Remember that the chapter is an interactive Jupyter notebook. If something is confusing or you have a question, you can insert a new cell and run your own code to see what happens.

#### Answer the following written questions

There is a blank Markdown cell after each question for your answer (double click in the blank cell to type your answer). Be sure to run your Markdown cells to format your answers.

1. How does the book define topology? How does this relate to how topology was discussed in your other GIS course(s)?

2. What is meant by a "sparse" representation of a weights matrix? What is the advantage of this way of storing the weights information?

3. In the initial book examples, the authors are working with a 3x3 grid of cells in space. However, the non-sparse representation of the weights matrix for this grid is 9x9. Why is this?

4. What is meant by the idea that relationships in time are "unidirectional", but relationships in space are "bidirectional"? 

5. Link the three concepts of "geographic tables", "spatial graphs" and "spatial weights." (Note: this question is not asking for a definition of each concept, it is asking for how these three concepts relate to one another. Hint: you might want to review Chapters 1 and 3.)

6. The following sentence from the book has a typo: "More specifically, polygons $0$ and $5$ are not Rook neighbors, but they do in fact share a common border." What needs to be fixed in this sentence? (Hint: fixing a typo means changing one small thing, not rewriting the sentence.)

7. For each type of weights listed below state whether weights are "binary" (i.e., can only take values of zero or one) or "continuous" (i.e., can take any value). (Note: enter your answer by editing the cell below.)

- Rook:
- Queen:
- k nearest neighbors: 
- Kernel:  
- Distance band (basic):  
- Distance band (hybrid):  
- Block: 

8. Why do distance based spatial weights need to be measured using projected data or great circle distance? Why does this not matter for contiguity based spatial weights?

9. The authors mentioned that rook weights are symmetrical. This means that if Polygon5 is a rook neighbor of Polygon7, then it is necessarily true that Polygon5 is a rook neighbor of Polygon7. However, k nearest neighbor weights are not necessarily symmetrical. If we identify the 4 nearest neighbors (KNN4) for each polygon and find that Polygon5 is a KNN4 neighbor of Polygon7, then Polygon7 may or may not be a KNN4 neighbor of Polygon5. Explain why rook weights are symmetrical and k nearest neighbor weights are not necessarily symmetrical. (Hint: drawing some polygons with pen and paper might make these ideas more concrete.)

#### The following questions require you to run Python code.

The cell below imports the GeoPandas package and the standard extra stuff to make things run smoother. Run this cell. 

In [1]:
import warnings
warnings.filterwarnings("ignore")

import geopandas as gpd

10. The Town of Cary, North Carolina provides spatial data on their trash collection services. You can see the metadata and other information at the following link.
https://data.townofcary.org/explore/dataset/solid-waste-and-recycling-collection-routes

    Run the following cell. The first two lines read in census tract and town data from the US Census Bureau. The lines labeled `L1` to `L6` read in the trash collection data from the town. After running the cell, explain what each line from `L1` to `L6` is doing.

In [ ]:
tracts = gpd.read_file('data/cary_tracts.geojson')
cary = gpd.read_file('data/cary.geojson')

trash = gpd.read_file('data/cary_trash_collection_zones.geojson') #L1
trash = trash.loc[~trash.geometry.isna(),] #L2
trash = trash.explode() #L3
trash = trash.reset_index() #L4
trash = trash.drop(columns=['index','geo_shape']) #L5
trash = trash.to_crs(tracts.crs) #L6

11. Starting from the output of the previous question, create a new `trash` GeoDataFrame, that has the `square_miles` column removed. Why is the `square_miles` column no longer relevant? (Hint: look at the steps from the previous code cell.)

12. Run the following cell to see how the city is split by recycling week (see the metadata if you have questions). After running the cell, create a single map showing how the city is split by day of the week. (Note: make your map large like the recycling map and pick some nice colors and don't reuse the recycling colors https://matplotlib.org/stable/gallery/color/named_colors.html#css-colors.)

In [ ]:
ax = trash.loc[trash.cycle=='Blue',].plot(color='cornflowerblue', figsize=(10,10))
trash.loc[trash.cycle=='Yellow',].plot(color='goldenrod', ax=ax)
cary.boundary.plot(linewidth=1, color='black', ax=ax);

13. Create rook spatial weights for the Cary `tracts`. Then plot the resulting spatial relationships on a map similar to the San Diego map in the book. (Note: the textbook plot has two maps so the code is a little more complicated than what is being asked here. Note: the map should be large like the previous plots.)

14. Look closely at the map you produced for the previous question. How many MORE lines do you think will be on the map for the queen case? Write down your answer now.

    - Create queen spatial weights for `tracts`.
    - Reproduce the map above (tract outlines and rook connections), but this time plot the queen connections too. Rook should be the last layer added and use different color lines from queen connections.
    - Report the value of the `nonzero` attribute for the queen and rook weights.
    - In a Markdown cell, discuss whether you over or under estimated the number of additional linkages, and why.

15. Create kernel spatial weights for `tracts`.
    - When creating the weights, explicitly pass in a value to the `bandwidth` parameter.
    - Make the same style map as above, this time with just the kernel weight connections and the tract outlines.
    - Explain what the `bandwidth` parameter does, its units and why you chose the particular value you did. Do all the tracts have neighbors? Why or why not?

16. Create queen spatial weights for `trash`. What percentage of the `trash` polygons are islands (when using queen weights)? What percentage of the `tracts` polygons are islands (when using queen weights)? Explain why the two percentages are so different. (Note: you need to compute the percentage being requested here, it is different from `pct_nonzero` discussed in the book. Hint: `trash.explore()` might help answer the "why" question.)

17. Create block spatial weights for `trash` based on day of the week. Report `nonzero` and `pct_nonzero` on the block weights result and the queen weights result for `trash`. Why is the block result so much more connected than the queen result? (Note: the book example for block weights creation uses an _optional_ `ids` parameter, you do not need it.)

18. In an earlier question, you explored how bandwidth works in relation to kernel weights. Here you will explore the kernel function options. 

    The first cell below computes kernel weights for `trash` using the default kernel function (triangular) and a bandwidth of 500. It shows that for this bandwidth you get over 180,000 connections.

    The second cell has a lot of code. The first function (`get_dists`) computes the distance between each of the 180,000 pairs of polygons. The second function (`plot_kernel`) creates a scatter plot of the distance between the 180,000 pairs (on the x axis) and the weight between the 180,000 pairs (on the y axis). You do not need to understand the code itself, just the output of the `plot_kernel` function.
   
    The last four cells run `plot_kernel` for four different kernel functions. A mathematical description of each kernel can be found at: https://pysal.org/libpysal/generated/libpysal.weights.Kernel.html.
   
    Run all the cells and then describe how the weight differs by distance for each kernel function. Your answer should include a description of each kernel's shape and the implied importance the user is putting on different distances if they were to choose that particular kernel. (Hint: they are all different. Also, do not say something like, "weights change quadratically" since we already know this from the kernel name. Your descriptions should be for a non-technical audience.)

In [ ]:
wk_trash = weights.distance.Kernel.from_dataframe(trash, bandwidth=500)
wk_trash.nonzero

In [2]:
import pandas as pd

# function that computes the distances between polygons up to a specified bandwidth
def get_dists(bw):
    w_kernel = weights.distance.Kernel.from_dataframe(trash, bandwidth=bw)   # triangular weights
    dists = {}                       # empty dictionary to hold distances
    for focal_id, wghts in w_kernel.weights.items():   # iterate over the focal IDs
        dists_temp = []              # holder for distance values
        for weight in wghts:         # iterate over the weights for a particular focal ID
            dist = bw * (1 - weight) # convert triangular weight to distance
            dists_temp.append(dist)  # add distance to holder list
        dists[focal_id] = dists_temp # store all distances for a particular focal ID
    return dists

# function that plots weights against distance for a particular type of kernel
def plot_kernel(bw, kernel, dists):
    dist_weight = []                                   # holder of distances and weights
    w_kernel = weights.distance.Kernel.from_dataframe(trash, bandwidth=bw, function=kernel)   # get weights
    for focal_id, wghts in w_kernel.weights.items():   # iterate over focal IDs
        pairs = zip(dists[focal_id], wghts)            # align distances to corresponding weights
        dist_weight.extend(pairs)                      # add the distance weight pairs to the holder 
    dist_weight_df = pd.DataFrame(dist_weight, columns=['dist','weight'])   # create DataFrame for plotting
    dist_weight_df.plot.scatter(x='dist', y='weight'); # create a scatter plot

In [ ]:
bw = 500
dists = get_dists(bw)
plot_kernel(bw, 'uniform', dists)

In [ ]:
bw = 500
dists = get_dists(bw)
plot_kernel(bw, 'triangular', dists)

In [ ]:
bw = 500
dists = get_dists(bw)
plot_kernel(bw, 'quadratic', dists)

In [ ]:
bw = 500
dists = get_dists(bw)
plot_kernel(bw, 'quartic', dists)